# pMF-B/32 no Colab T4 · LoRA vs. DGL

Seleciona um kernel Colab GPU. Corre as células pela ordem indicada. O smoke faz quatro updates por abordagem, um por tarefa, e mostra as amostras e os resultados. Reutiliza o dataset e os pesos em cache. Se for um runtime novo, ativa os downloads explicitamente na primeira célula.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/Synthetic-Data-Generation-Framework")
DATASET = Path("/content/rf100-bi3q3")
OUTPUT = Path("/content/pmf_lora_smoke")
BACKBONE = None  # Ou Path("/content/.../transformer") para pesos já disponíveis.
ALLOW_DOWNLOADS = True
REPO_URL = "https://github.com/andreferraria453/Synthetic-Data-Generation-Framework.git"

subprocess.run(["nvidia-smi", "-L"], check=True)
if REPO.exists():
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

try:
    import diffusers
    from huggingface_hub import snapshot_download
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "diffusers==0.35.2", "accelerate==1.10.1"], check=True)
    from huggingface_hub import snapshot_download

if not (DATASET / "data.yaml").is_file():
    if not ALLOW_DOWNLOADS:
        raise FileNotFoundError(f"Indica o export RF100 em DATASET ({DATASET}) ou ativa ALLOW_DOWNLOADS.")
    snapshot_download(
        repo_id="LibreYOLO/corrosion-bi3q3", repo_type="dataset",
        revision="d20b5b1e6b21ff7de555bbf848dcf7d30ccf8ee7", local_dir=str(DATASET),
    )

if BACKBONE is None:
    try:
        snapshot = snapshot_download(
            repo_id="BiliSakura/pMF-diffusers",
            revision="ee50c8763200f26b0656c396315909b762be6191",
            allow_patterns=["pMF-B-32/transformer/*"],
            local_files_only=not ALLOW_DOWNLOADS,
        )
    except OSError as error:
        raise RuntimeError("Falta o pMF-B/32 em cache. Indica BACKBONE ou ativa ALLOW_DOWNLOADS.") from error
    BACKBONE = Path(snapshot) / "pMF-B-32" / "transformer"

for name in ("config.json", "transformer_pmf.py", "diffusion_pytorch_model.safetensors"):
    if not (BACKBONE / name).is_file():
        raise FileNotFoundError(f"Falta {name} em {BACKBONE}. Indica BACKBONE ou ativa ALLOW_DOWNLOADS.")

In [ ]:
from collections import deque

command = [
    sys.executable, "-u", "-m", "rae_experiment.any_to_any.compare_lora",
    "--dataset", str(DATASET), "--output", str(OUTPUT), "--backbone", str(BACKBONE),
    "--steps", "50", "--interval", "4", "--batch-size", "2", "--device", "cuda",
]
tail = deque(maxlen=25)
with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, errors="replace") as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line)
    except KeyboardInterrupt:
        process.terminate()
        raise
    status = process.wait()
if status:
    raise RuntimeError("".join(tail))

In [ ]:
import json
from PIL import Image
from IPython.display import display

tasks = ("rgb_to_boxes", "boxes_to_rgb", "noise_to_rgb_boxes", "noise_to_rgb")
for approach in ("A_shared", "B_dgl"):
    directory = OUTPUT / approach
    protocol = json.loads((directory / "protocol.json").read_text())
    report = json.loads((directory / "0004.json").read_text())
    print(approach, json.dumps({
        "parâmetros treináveis": protocol["trainable"],
        "paridade RGB e JVP": protocol["rgb_audit"],
        "losses": report["loss"],
        "norma de gradiente (máx.)": report["gradient_norm_max"],
        "boxes": report["inference"]["rgb_to_boxes"],
        "treino (s)": report["train_seconds"],
    }, ensure_ascii=False, indent=2))
    images = [Image.open(directory / "0004" / f"{task}.png") for task in tasks]
    width, height = images[0].size
    grid = Image.new("RGB", (2 * width, 2 * height))
    for i, image in enumerate(images):
        grid.paste(image, (i % 2 * width, i // 2 * height))
        image.close()
    display(grid)